<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Daily_Challenge_MCP_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: MCP Ping-Pong — Weather Demo
This notebook builds a minimal MCP server and client without any LLM calls. The client launches the server over **STDIO**, discovers its resource and tool, reads the supported-city list, and calls the weather tool.

## 1. Setup
Install the MCP SDK and CLI. The version is pinned to the stable `1.x` line so the exercise keeps the API expected by the starter notebook.

In [ ]:
# Install the stable MCP Python SDK and CLI
%pip install -qU "mcp[cli]>=1.27,<2"

In [ ]:
# Verify the environment
!python --version
!mcp version
!mcp --help | head -n 12

## 2. MCP server: `server.py`
The server is named `WeatherDemo`. It exposes:
- `get_weather(city: str)`: a tool returning static weather data.
- `cities://list`: a text resource containing one supported city per line.

In [ ]:
%%writefile server.py
import logging
from mcp.server.fastmcp import FastMCP

# Logs are written to stderr, so they do not corrupt MCP's STDIO messages.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("WeatherDemo")

mcp = FastMCP("WeatherDemo")

CITY_DATA = {
    "paris": {
        "city": "Paris",
        "temp_c": 21,
        "condition": "sunny",
    },
    "london": {
        "city": "London",
        "temp_c": 18,
        "condition": "cloudy",
    },
    "nyc": {
        "city": "NYC",
        "temp_c": 24,
        "condition": "breezy",
    },
}


@mcp.tool()
def get_weather(city: str) -> dict:
    """Return static weather information for a supported city."""
    key = city.strip().lower()
    logger.info("get_weather called for city=%s", city)

    data = CITY_DATA.get(key)
    if data is None:
        return {
            "error": f"Unsupported city: {city}",
            "supported_cities": [
                item["city"] for item in CITY_DATA.values()
            ],
        }

    return data.copy()


@mcp.resource("cities://list")
def list_cities() -> str:
    """Return the supported cities, one city per line."""
    return "\n".join(
        item["city"] for item in CITY_DATA.values()
    )


if __name__ == "__main__":
    # STDIO is FastMCP's default transport.
    mcp.run()


## 3. MCP client: `client.py`
The client launches `server.py` with the MCP CLI over STDIO, initializes a `ClientSession`, discovers capabilities, reads the resource, and invokes the tool.

In [ ]:
%%writefile client.py
import asyncio

from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from pydantic import AnyUrl


server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None,
)


async def main() -> None:
    """Connect to the weather server and exercise its capabilities."""
    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            resources = await session.list_resources()
            print("Resources:")
            for resource in resources.resources:
                print(f"- {resource.uri}")

            tools = await session.list_tools()
            print("\nTools:")
            for tool in tools.tools:
                print(f"- {tool.name}")

            cities_result = await session.read_resource(
                AnyUrl("cities://list")
            )
            cities_content = cities_result.contents[0]

            print("\ncities://list:")
            if isinstance(cities_content, types.TextResourceContents):
                print(cities_content.text)
            else:
                print(cities_content)

            weather_result = await session.call_tool(
                "get_weather",
                arguments={"city": "Paris"},
            )

            print("\nget_weather('Paris'):")
            if weather_result.structuredContent:
                print(weather_result.structuredContent)
            else:
                for item in weather_result.content:
                    if isinstance(item, types.TextContent):
                        print(item.text)
                    else:
                        print(item)


if __name__ == "__main__":
    asyncio.run(main())


## 4. Run the end-to-end demo
Running the client is enough: it automatically starts and communicates with the server.

In [ ]:
!python client.py

Resources:
- cities://list

Tools:
- get_weather

cities://list:
Paris
London
NYC

get_weather('Paris'):
{
  "city": "Paris",
  "temp_c": 21,
  "condition": "sunny"
}


## 5. Expected result and submission
The output above must show:
- the `cities://list` resource;
- the `get_weather` tool;
- the cities Paris, London, and NYC;
- the weather response for Paris.

Submit `server.py`, `client.py`, and a terminal capture. Then place the files in your GitHub repository and push them.